# Chapter 13: Stock Exchange
*System Design Interview Volume 2*

## TL;DR

A stock exchange matches buy and sell orders through a **matching engine** backed by a per-symbol **order book** (doubly-linked list at each price level for O(1) place/match/cancel). A **sequencer** stamps every order and execution with sequential IDs, enabling **deterministic replay** and exactly-once guarantees. For ultra-low latency, modern exchanges run all critical-path components on a **single server** communicating via **mmap event stores** (no network, no disk I/O). **Event sourcing** records every state change as an immutable event, powering hot-warm failover with **Raft** consensus across replicas. The **market data publisher** rebuilds order books and candlestick charts from the execution stream and distributes them via **reliable UDP multicast** for fairness.

## Requirements

| Type | Requirement | Detail |
|---|---|---|
| Functional | Place/cancel limit orders | Clients submit orders via brokers |
| Functional | Match orders | Matching engine pairs buy/sell orders in real-time |
| Functional | Real-time order book | L1/L2/L3 market data for subscribers |
| Functional | Executions & market data | Stream matched fills to brokers and data services |
| Functional | Risk checks | Daily volume caps, wallet fund verification |
| Non-functional | Availability | 99.99% (8.64s downtime/day max) |
| Non-functional | Latency | Millisecond-level round-trip (99th percentile focus) |
| Non-functional | Fault tolerance | Immediate recovery via event replay |
| Non-functional | Security | KYC, DDoS protection, rate limiting |

## Estimation: Exchange Throughput

In [ ]:
# Back-of-envelope for stock exchange
symbols = 100
orders_per_day = 1_000_000_000  # 1 billion

# NYSE trading hours: 9:30 AM - 4:00 PM ET = 6.5 hours
trading_hours = 6.5
trading_seconds = trading_hours * 3600
print(f"Trading window: {trading_hours} hours = {trading_seconds:,.0f} seconds")

# QPS
avg_qps = orders_per_day / trading_seconds
peak_qps = 5 * avg_qps  # morning open + afternoon close spikes
print(f"\nAverage QPS: {avg_qps:,.0f}")
print(f"Peak QPS (5x): {peak_qps:,.0f}")

# Latency budget on critical path
# gateway -> order manager -> sequencer -> matching engine
network_hops = 3
network_rtt_us = 500  # microseconds per hop
total_network_us = network_hops * network_rtt_us
print(f"\nNetwork latency ({network_hops} hops x {network_rtt_us}us): {total_network_us:,} us")
print(f"  = {total_network_us / 1000:.1f} ms")

# With mmap on single server: sub-microsecond IPC
mmap_latency_us = 0.5
total_mmap_us = network_hops * mmap_latency_us
print(f"\nmmap IPC latency ({network_hops} hops x {mmap_latency_us}us): {total_mmap_us} us")
print(f"  Speedup: {total_network_us / total_mmap_us:,.0f}x faster")

# High availability budget
availability = 0.9999
downtime_sec_per_day = (1 - availability) * 86400
print(f"\n99.99% availability = {downtime_sec_per_day:.2f}s downtime/day")
print("Requires automatic failover (no human intervention)")

## High-Level Design

```mermaid
graph TB
    subgraph Clients["Brokers / Clients"]
        B1["Robinhood"]
        B2["Goldman Sachs"]
        B3["Colo Engine"]
    end

    subgraph Exchange["Stock Exchange -- Critical Trading Path"]
        GW["Client Gateway<br/>Auth, Validation, Rate Limit, FIX"]
        OM["Order Manager<br/>Risk Checks, Wallet, State"]
        SEQ_IN["Inbound Sequencer"]
        ME["Matching Engine<br/>Order Book per Symbol"]
        SEQ_OUT["Outbound Sequencer"]
    end

    subgraph MarketData["Market Data Flow"]
        MDP["Market Data Publisher<br/>Order Books, Candlesticks"]
        DS["Data Service"]
    end

    subgraph Reporting["Reporting Flow"]
        RPT["Reporter"]
        DB[("Reporting DB")]
    end

    B1 & B2 & B3 -->|orders| GW
    GW --> OM
    OM --> SEQ_IN
    SEQ_IN --> ME
    ME --> SEQ_OUT
    SEQ_OUT --> OM
    OM --> GW
    GW -->|executions| B1 & B2 & B3

    ME -->|execution stream| MDP
    MDP --> DS
    DS -->|market data| B1 & B2 & B3

    OM -->|orders + fills| RPT
    RPT --> DB
```

### Critical Path Flow

1. Client places limit order via broker
2. **Client gateway** validates, authenticates, rate-limits, normalizes (FIX protocol)
3. **Order manager** runs risk checks (daily volume cap) and verifies wallet funds
4. **Inbound sequencer** stamps order with sequential ID
5. **Matching engine** processes order against the order book
6. If matched: two executions (fills) emitted, stamped by **outbound sequencer**
7. Executions returned to client via order manager and gateway

## Deep Dive

### Order Book Data Structure

```mermaid
graph LR
    subgraph OB["Order Book for AAPL"]
        subgraph SellBook["Sell Book (Asks)"]
            S3["100.13: 100, 200"]
            S2["100.12: ..."]
            S1["100.11: ..."]
            S0["Best Ask 100.10: 200, 400, 1100, 100"]
        end
        subgraph BuyBook["Buy Book (Bids)"]
            B0["Best Bid 100.08: 500, 600, 900"]
            B1["100.07: ..."]
            B2["100.06: ..."]
            B3["100.05: 500, 100"]
        end
    end

    subgraph Helpers["Helper Structures"]
        LM["limitMap: Price -> PriceLevel<br/>O(1) price lookup"]
        OM2["orderMap: OrderID -> Order<br/>O(1) cancel lookup"]
    end
```

**Key operations -- all O(1):**
- **Place**: append Order to tail of PriceLevel's doubly-linked list
- **Match**: remove Order from head of PriceLevel's doubly-linked list
- **Cancel**: find via orderMap, remove via doubly-linked list prev/next pointers

### Matching Algorithm (FIFO / Price-Time Priority)

In [ ]:
# Simplified FIFO matching algorithm demonstration
from collections import OrderedDict

class Order:
    def __init__(self, order_id, side, price, quantity):
        self.order_id = order_id
        self.side = side          # 'BUY' or 'SELL'
        self.price = price
        self.quantity = quantity
        self.matched_qty = 0

    @property
    def remaining(self):
        return self.quantity - self.matched_qty

    def __repr__(self):
        return f"Order({self.order_id}, {self.side}, ${self.price}, qty={self.remaining})"


def match_order(buy_order, sell_book):
    """Match a buy order against the sell book (ascending price levels)."""
    fills = []
    remaining = buy_order.remaining

    for price in sorted(sell_book.keys()):
        if price > buy_order.price or remaining <= 0:
            break
        queue = sell_book[price]
        while queue and remaining > 0:
            sell_order = queue[0]
            matched = min(remaining, sell_order.remaining)
            remaining -= matched
            sell_order.matched_qty += matched
            buy_order.matched_qty += matched
            fills.append((buy_order.order_id, sell_order.order_id, price, matched))
            if sell_order.remaining == 0:
                queue.pop(0)
        if not queue:
            del sell_book[price]

    return fills

# Simulate an order book scenario
sell_book = {
    100.10: [Order("S1", "SELL", 100.10, 200),
             Order("S2", "SELL", 100.10, 400)],
    100.11: [Order("S3", "SELL", 100.11, 300)],
}

incoming = Order("B1", "BUY", 100.11, 700)
print(f"Incoming: {incoming}")
print(f"Sell book before: { {p: [str(o) for o in orders] for p, orders in sell_book.items()} }")

fills = match_order(incoming, sell_book)

print(f"\nFills:")
for buy_id, sell_id, price, qty in fills:
    print(f"  {buy_id} <-> {sell_id} @ ${price} x {qty} shares")
print(f"\nBuy order remaining: {incoming.remaining}")
print(f"Sell book after: { {p: [str(o) for o in orders] for p, orders in sell_book.items()} }")

### Event Sourcing with mmap

```mermaid
graph TB
    subgraph SingleServer["Single Server -- All Critical Path"]
        subgraph Components["Components (each pinned to a CPU core)"]
            GW2["Gateway"]
            OM2["Order Manager"]
            ME2["Matching Engine"]
            MDP2["Market Data Publisher"]
        end

        subgraph EventStore["mmap Event Store (/dev/shm)"]
            ES["Immutable Event Log<br/>NewOrderEvent, OrderFilledEvent<br/>Sub-microsecond IPC"]
        end

        subgraph RingBuffers["Per-Component Ring Buffers"]
            RB1["Gateway Ring Buffer"]
            RB2["ME Ring Buffer"]
        end

        SEQ2["Sequencer<br/>(Single Writer)"]
    end

    GW2 -->|write| RB1
    ME2 -->|write| RB2
    SEQ2 -->|pull from| RB1
    SEQ2 -->|pull from| RB2
    SEQ2 -->|stamp + write| ES
    ES -->|read| OM2
    ES -->|read| ME2
    ES -->|read| MDP2
```

**Why event sourcing for exchanges:**
- **Deterministic replay**: given same events in same order, identical state is reproduced
- **Fast recovery**: warm instance replays event log to catch up instantly
- **Audit trail**: immutable log satisfies regulatory requirements
- **No network/disk on critical path**: mmap over /dev/shm = memory-only access

### Hot-Warm Failover with Raft

```mermaid
graph TB
    subgraph RaftCluster["Raft Cluster (5 nodes)"]
        L["Leader (Hot)<br/>Processes + emits events"]
        F1["Follower 1 (Warm)<br/>Processes events, emits nothing"]
        F2["Follower 2 (Warm)"]
        F3["Follower 3 (Warm)"]
        F4["Follower 4 (Warm)"]
    end

    L -->|"AppendEntries RPC"| F1
    L -->|"AppendEntries RPC"| F2
    L -->|"AppendEntries RPC"| F3
    L -->|"AppendEntries RPC"| F4

    L -->|"Heartbeat"| F1 & F2 & F3 & F4

    F1 -.->|"If leader fails: RequestVote"| F2 & F3 & F4
    F1 -.->|"Majority vote -> new leader"| F1
```

**Recovery objectives:**
- **RTO (Recovery Time Objective)**: seconds -- automatic failover, no human intervention
- **RPO (Recovery Point Objective)**: near zero -- Raft guarantees consensus before acknowledging

### Market Data Publisher

```mermaid
graph LR
    subgraph MDP3["Market Data Publisher"]
        OB1["Order Book<br/>Sym 1"]
        OB2["Order Book<br/>Sym 2"]
        OBN["Order Book<br/>Sym N"]
        CS["Candlestick Charts<br/>1min, 5min, 1hr, 1day"]
        RB["Ring Buffer<br/>Recent 100 ticks"]
    end

    ME3["Matching Engine"] -->|"execution stream"| MDP3
    MDP3 -->|"L1/L2/L3 data"| DS2["Data Service"]
    MDP3 -->|"persist"| KDB["KDB+ Columnar DB"]
    DS2 -->|"Reliable UDP Multicast"| SUB["All Subscribers<br/>(simultaneous delivery)"]
```

**Fairness**: reliable UDP multicast delivers data to all subscribers at the same time, preventing any single client from having a latency advantage.

## Trade-offs

| Dimension | Option A | Option B |
|---|---|---|
| Single server (all components) | Sub-microsecond IPC via mmap, no network hops | Single point of failure, vertical scaling limit |
| Distributed microservices | Independent scaling, fault isolation | Network latency adds up to milliseconds |
| mmap event store | Memory-speed IPC, no disk on critical path | Complex to implement, OS-specific |
| Kafka as event store | Battle-tested, rich ecosystem | Higher and less predictable latency |
| FIFO matching | Simple, fair, well-understood | No priority for market makers |
| FIFO + LMM allocation | Incentivizes liquidity providers | Complex, less transparent |
| Single-threaded app loop | No locks, no context switches, predictable latency | Single core throughput ceiling |
| Multi-threaded engine | Higher throughput potential | Lock contention, unpredictable 99p latency |
| Hot-warm failover | Fast recovery, simple model | Only works within one server |
| Raft cluster (multi-server) | Cross-datacenter fault tolerance | Replication latency, split-brain risk |

## Key Takeaways

1. **Determinism is foundational**: the sequencer + event sourcing guarantee that replaying events reproduces identical state, enabling recovery and auditing
2. **Minimize the critical path**: only gateway, order manager, sequencer, and matching engine; even logging is removed
3. **Eliminate network and disk I/O**: mmap over /dev/shm gives sub-microsecond IPC; single-server design removes network hops entirely
4. **O(1) order book operations**: doubly-linked lists per price level + hash maps for price and order lookup
5. **Single-threaded + CPU pinning**: no context switches, no locks, predictable 99th percentile latency
6. **Event sourcing enables hot-warm failover**: warm instance processes all events but emits nothing; takes over instantly when primary fails
7. **Raft consensus for multi-server HA**: replicate event store across 5+ nodes; automatic leader election on failure
8. **Fair market data distribution**: reliable UDP multicast delivers to all subscribers simultaneously

## Related Concepts

- [[matching-engine]] -- FIFO algorithm, deterministic matching, single-threaded design
- [[order-book]] -- doubly-linked list per price level, O(1) operations
- [[sequencer]] -- sequential IDs, exactly-once, event store integration
- [[market-data-publisher]] -- order book rebuild, candlestick charts, ring buffers, multicast
- [[event-sourcing-exchange]] -- mmap event store, hot-warm failover, Raft replication
- [[risk-checks-order-management]] -- order manager, risk checks, wallet, client gateway